# TRATA RELEVÂNCIA VOTAÇÕES

In [ ]:
import pandas as pd
import unicodedata
import re

# =================================================================
# DICIONÁRIO DE PESOS
# =================================================================
PESOS_TEMAS = {

    # PESO 5 (Alto Impacto / Mudanças Estruturais)
    'CODIGO PENAL': 5, 'CONSTITUICAO': 5, 'MULHER': 5, 'CRIANCA': 5,
    'SEGURIDADE SOCIAL': 5, 'SAUDE': 5, 'EDUCACAO': 5, 'TRIBUTACAO': 5,
    'IMPOSTO': 5, 'CRIME': 5, 'DIREITOS HUMANOS': 5, 'MEIO AMBIENTE': 5,
    'DEFICIENCIA': 5, 'DEFICIENTE': 5, 'PESSOA COM DEFICIENCIA': 5, 'ACESSIBILIDADE': 5, 'INCLUSÃO': 5,

    # PESO 3 (Impacto Médio / Regulamentações e Direitos Trabalhistas)
    'TRABALHO': 3, 'JORNADA': 3, 'TRANSPORTE': 3, 'FARMACIA': 3,
    'NUTRICAO': 3, 'SEGURO PRIVADO': 3, 'CONSUMIDOR': 3, 'COMERCIO': 3,
    'NUTRICIONISTA': 3, 'ALIMENTACAO': 3, 'AGRICULTURA': 3, 'PECUARIA': 3, 'AGROPECUARIA': 3,

    # PESO 1 (Baixo Impacto / Simbólico ou Administrativo)
    'HOMENAGEM': 1, 'DIA NACIONAL': 1, 'COMEMORACAO': 1, 'DENOMINACAO': 1,
    'NOME': 1, 'PLACA': 1, 'DATA COMEMORATIVA': 1
}


PESO_PADRAO = 2

# =================================================================
# 2. FUNÇÃO DE LIMPEZA DE TEXTO (NLP Básico)
# =================================================================
def limpar_texto(texto):
    if pd.isna(texto):
        return ""


    texto = unicodedata.normalize('NFKD', str(texto)).encode('ASCII', 'ignore').decode('utf-8')


    texto = texto.upper()


    texto = re.sub(r'[^A-Z0-9\s,]', '', texto)

    return texto.strip()

# =================================================================
# 3. FUNÇÃO PARA CALCULAR O PESO DA PROPOSIÇÃO
# =================================================================
def definir_peso_projeto(keywords_limpas):
    if pd.isna(keywords_limpas) or not keywords_limpas:
        return 2


    palavras_peso_1 = ['DIA NACIONAL', 'HOMENAGEM', 'DENOMINACAO', 'DATA COMEMORATIVA', 'NOME', 'PLACA']
    for palavra in palavras_peso_1:
        if palavra in keywords_limpas:
            return 1


    peso_final = 0

    for tema, peso in PESOS_TEMAS.items():
        if tema in keywords_limpas:
            peso_final = max(peso_final, peso)


    if peso_final == 0:
        return 2

    return peso_final

# =================================================================
# --- EXECUÇÃO PRINCIPAL ---
# =================================================================
print(" Iniciando Módulo 1.5 - Limpeza e Pesos das Keywords...")

try:

    df = pd.read_csv("modulo1_keywords.csv")


    print("🧹 Limpando caracteres especiais e acentos...")
    df['keywords_limpas'] = df['keywords'].apply(limpar_texto)


    print("⚖️ Calculando peso de cada proposição...")
    df['peso_projeto'] = df['keywords_limpas'].apply(definir_peso_projeto)


    df.to_csv("modulo1_5_projetos_com_pesos.csv", index=False)


    print(" Agrupando pontuação de impacto por deputado...")
    impacto_deputados = df.groupby('deputado_id').agg(
        pontuacao_total_impacto=('peso_projeto', 'sum'),
        media_peso_projetos=('peso_projeto', 'mean')

    ).reset_index()


    impacto_deputados['media_peso_projetos'] = impacto_deputados['media_peso_projetos'].round(2)


    impacto_deputados.to_csv("modulo1_5_impacto_pronto.csv", index=False)

    print("\n Módulo 1.5 Concluído! Arquivo 'modulo1_5_impacto_pronto.csv' gerado.")
    print("Exemplo dos resultados:")
    print(impacto_deputados.head())

except FileNotFoundError:
    print(" Erro: Arquivo 'modulo1_keywords.csv' não encontrado. Rode o Módulo 1 primeiro!")